Explanation provided in [link](https://stats.stackexchange.com/questions/459296/confusion-on-scikit-learn-nested-cross-validation-example) and this [sci-kit example](https://scikit-learn.org/stable/auto_examples/model_selection/plot_nested_cross_validation_iris.html#sphx-glr-auto-examples-model-selection-plot-nested-cross-validation-iris-py) helped develop this nested cv method.


Explanation provided in [link](https://pychemauth.readthedocs.io/en/latest/jupyter/learn/cv_optimization.html) discuss the role of nested cv in comparing different models, and how to fit the final model for ultimate performance measures. This [link](https://stats.stackexchange.com/questions/647319/is-my-understanding-approach-to-nested-cross-validation-final-model-tuning-corr) might also help in the same area.

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import loguniform, randint

from sklearn.preprocessing import PowerTransformer
from sklearn.neighbors import LocalOutlierFactor

from sklearn.model_selection import (
    StratifiedKFold,
    StratifiedShuffleSplit,
    RepeatedStratifiedKFold,
    RandomizedSearchCV,
    cross_validate
)

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn import FunctionSampler
from imblearn.over_sampling import SMOTE

from sklearn.svm import SVC

In [ ]:
# Load the dataset
df = pd.read_csv("Datasets/Botswana_GAD_eGeMAPS_QBF.csv")

# Modify GAD7 target for binary classification
df['Anxiety_Binary'] = df['GAD7_Total'].apply(lambda x: 1 if x >= 5 else 0)

# Define Metadata to Drop
metadata_cols = [
    'SessionID', 'QBF_Name', 'Sex', 'Age', 'Health', 'Health_Binary',
    'Country', 'GAD7_Total', 'Anxiety_Category', 'Anxiety_Binary'
]

# Define feature and target
X = df.drop(columns=metadata_cols, errors='ignore')
X = X.select_dtypes(include=[np.number])
y = df['Anxiety_Binary']

X_arr = X.to_numpy()
y_arr = y.to_numpy()

# Define outlier detection and removal
def lof_outlier_removal(X, y, n_neighbors=20, contamination=0.05, algorithm='auto', metric='manhattan'):
    X_arr = np.asarray(X)
    y_arr = np.asarray(y)

    lof = LocalOutlierFactor(
        n_neighbors=n_neighbors,
        contamination=contamination,
        algorithm=algorithm,
        metric=metric,
        leaf_size=30,
        novelty=False)
    
    y_pred = lof.fit_predict(X_arr)
    mask_inliers = y_pred == 1

    return X_arr[mask_inliers], y_arr[mask_inliers]

# Create sampler for outlier removal
lof_sampler = FunctionSampler(
    func=lof_outlier_removal,
    kw_args={
        'contamination': 0.05,
        'n_neighbors': 20,
        'algorithm': 'auto',
        'metric': 'manhattan'
    },
    validate=False)

# Define the pipeline
pipeline = ImbPipeline([
    ("yjpt", PowerTransformer(method='yeo-johnson', standardize=True)),
    ("outlier_removal", lof_sampler),
    ("oversampling", SMOTE(k_neighbors=5, random_state=42)),
    ("clf", SVC(probability=False, kernel='rbf', random_state=42)) 
])

# Define the hyperparameter search space
param_grid_rs = {
    'oversampling__k_neighbors': randint(3, 10),
    'clf__C': loguniform(1e-1, 1e6),
    'clf__gamma': loguniform(1e-5, 1e1),
}

# Define multiple scoring metrics
scoring = {
    'roc_auc': 'roc_auc',
    'balanced_accuracy': 'balanced_accuracy',
    'average_precision': 'average_precision',
    'f1_score': 'f1'
}

In [ ]:
# Nested Cross-Validation
NUM_REPETITION = 4
OUTER_SPLITS = 5
INNER_SPLITS = 3
N_ITER_SEARCH = 100

# Define outer and inner cross-validation strategies
outer_cv = RepeatedStratifiedKFold(n_splits=OUTER_SPLITS, n_repeats=NUM_REPETITION, random_state=42)
inner_cv = StratifiedKFold(n_splits=INNER_SPLITS, shuffle=True, random_state=42)
# inner_cv = StratifiedShuffleSplit(n_splits=INNER_SPLITS, random_state=42)

# Define the RandomizedSearchCV model
model = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_grid_rs,
    n_iter=N_ITER_SEARCH,
    scoring=scoring,
    refit='roc_auc',
    cv=inner_cv,
    return_train_score=True,
    n_jobs=1,
    verbose=1,
    random_state=42
)

# Execute nested cross-validation
results = cross_validate(
    model,
    X=X_arr,
    y=y_arr,
    cv=outer_cv,
    scoring=scoring,
    return_estimator=True,
    n_jobs=-1,
    verbose=1
)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 192 concurrent workers.


Fitting 3 folds for each of 100 candidates, totalling 300 fits
Fitting 3 folds for each of 100 candidates, totalling 300 fits
Fitting 3 folds for each of 100 candidates, totalling 300 fits
Fitting 3 folds for each of 100 candidates, totalling 300 fits
Fitting 3 folds for each of 100 candidates, totalling 300 fits
Fitting 3 folds for each of 100 candidates, totalling 300 fits
Fitting 3 folds for each of 100 candidates, totalling 300 fits
Fitting 3 folds for each of 100 candidates, totalling 300 fits
Fitting 3 folds for each of 100 candidates, totalling 300 fits
Fitting 3 folds for each of 100 candidates, totalling 300 fits
Fitting 3 folds for each of 100 candidates, totalling 300 fits
Fitting 3 folds for each of 100 candidates, totalling 300 fits
Fitting 3 folds for each of 100 candidates, totalling 300 fits
Fitting 3 folds for each of 100 candidates, totalling 300 fits
Fitting 3 folds for each of 100 candidates, totalling 300 fits
Fitting 3 folds for each of 100 candidates, totalling 3

[Parallel(n_jobs=-1)]: Done  15 out of  20 | elapsed:   26.0s remaining:    8.7s
[Parallel(n_jobs=-1)]: Done  20 out of  20 | elapsed:   26.3s finished


In [ ]:
# Process and save results
results_summary_df = pd.DataFrame({
    m: results[f"test_{m}"] for m in list(scoring.keys())
}).agg(["mean", "std"]).T

n_outer = OUTER_SPLITS
n_total = OUTER_SPLITS * NUM_REPETITION

outer_df = pd.DataFrame({
    "repeat": (np.arange(n_total) // n_outer) + 1,
    "outer_fold": (np.arange(n_total) % n_outer) + 1,
    **{m: results[f"test_{m}"] for m in list(scoring.keys())}
})

inner_df = pd.DataFrame([
    {
        "repeat": (i // OUTER_SPLITS) + 1,
        "outer_fold": (i % OUTER_SPLITS) + 1,
        "inner_best_score": est.best_score_,
        "inner_best_params": est.best_params_,
        "n_candidates": len(est.cv_results_["params"]),
    }
    for i, est in enumerate(results["estimator"])
])

results_summary_df.to_csv("results_summary.csv", index=True)
outer_df.to_csv("outer_cv_results.csv", index=False)
inner_df.to_csv("inner_cv_results.csv", index=False)

In [5]:
results_summary_df
# outer_df
# inner_df

,mean,std
roc_auc,0.463008,0.059140
balanced_accuracy,0.487642,0.028840
average_precision,0.537678,0.046783
f1_score,0.580595,0.091624


In [ ]:
# Code development based on the sci-kit learn example: Nested versus non-nested cross-validation
# nested_roc_aucs = np.zeros(NUM_REPETITION)
# nested_baccs = np.zeros(NUM_REPETITION)
# nested_aps = np.zeros(NUM_REPETITION)

# for i in range(NUM_REPETITION):
#     print(f"Starting repetition {i+1}/{NUM_REPETITION}...")
#     outer_cv = StratifiedKFold(n_splits=OUTER_SPLITS, shuffle=True, random_state=42+i)
#     inner_cv = StratifiedShuffleSplit(n_splits=INNER_SPLITS, shuffle=True, random_state=42+i)

#     clf = RandomizedSearchCV(
#         estimator=pipeline, 
#         param_distributions=param_grid_rs, 
#         n_iter=N_ITER_SEARCH,
#         scoring=scoring,
#         refit='roc_auc',
#         cv=inner_cv,
#         n_jobs=-1,
#         return_train_score=True,
#         verbose=1,
#         random_state=42+i
#     )

#     print(f"Starting nested cross-validation for trial {i+1}/{NUM_REPETITION}...")
#     nested_score = cross_validate(clf, X=X_arr, y=y_arr, cv=outer_cv, scoring=scoring)
#     nested_roc_aucs[i] = nested_score['test_roc_auc'].mean()
#     nested_baccs[i] = nested_score['test_balanced_accuracy'].mean()
#     nested_aps[i] = nested_score['test_average_precision'].mean()
#     print(f"Trial {i+1}/{NUM_REPETITION} completed. Nested CV score: {nested_roc_aucs[i]:.4f}")

Starting repetition 1/4...
Starting nested cross-validation for trial 1/4...
Fitting 3 folds for each of 100 candidates, totalling 300 fits
Fitting 3 folds for each of 100 candidates, totalling 300 fits
Fitting 3 folds for each of 100 candidates, totalling 300 fits
Fitting 3 folds for each of 100 candidates, totalling 300 fits
Fitting 3 folds for each of 100 candidates, totalling 300 fits
Trial 1/4 completed. Nested CV score: 0.4725
Starting repetition 2/4...
Starting nested cross-validation for trial 2/4...
Fitting 3 folds for each of 100 candidates, totalling 300 fits
Fitting 3 folds for each of 100 candidates, totalling 300 fits
Fitting 3 folds for each of 100 candidates, totalling 300 fits
Fitting 3 folds for each of 100 candidates, totalling 300 fits
Fitting 3 folds for each of 100 candidates, totalling 300 fits
Trial 2/4 completed. Nested CV score: 0.4987
Starting repetition 3/4...
Starting nested cross-validation for trial 3/4...
Fitting 3 folds for each of 100 candidates, total